# A FAIRE POUR LANCER UN GROS ENTRAINEMENT

- utiliser un modèle b3 (ou +) que b0
- en csq changer le target size dans le preprocessor
- augmenter nombre d'epoch
- batch size ?

Optionnel :
- tester une autre loss

# **0. Librairies**

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import sys
import os


sys.path.append(os.path.abspath(".."))

from lib.data.dataset import BeeDataset

from lib.data.preprocessing import TorchPreprocessor

from lib.data.train_val_split import train_val_split

from lib.data.preprocessing import TorchPreprocessor

from lib.data.data_augmentation import data_augmented_loader

In [ ]:
## CONFIGURATION
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 32
NUM_EPOCHS = 25
LR = 1e-4

DROPOUT = 0.4
WEIGHT_DECAY = 1e-4

NUM_CLASSES = 50

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

## **1. Preprocessing**

In [4]:
import lib.data.preprocessing as prep
print(prep.__file__)

/home/alexandre-tonon/SDD/Hackathons/Hackaton_abeilles_tigres/lib/data/preprocessing.py


In [ ]:
train_loader, val_loader = data_augmented_loader(IMAGENET_MEAN, IMAGENET_STD, Diffference = False)

Train prêt : 6417 images (avec augmentation ciblée)
Val prête  : 1582 images (sans augmentation)


In [6]:
import pandas as pd

# --- Récupérer tous les labels du val_loader ---
all_labels = []

for _, labels in val_loader:
    all_labels.extend(labels.numpy() if isinstance(labels, torch.Tensor) else labels)

# --- Compter le nombre d'images par classe ---
class_counts_val = pd.Series(all_labels).value_counts().sort_index()

# --- Créer un DataFrame ---
df_val_counts = pd.DataFrame({
    "class": class_counts_val.index,
    "num_images": class_counts_val.values
})

# --- Sauvegarder dans un CSV ---
csv_val_path = "val_class_counts.csv"
df_val_counts.to_csv(csv_val_path, index=False)
print(f"CSV des images par classe dans la validation sauvegardé dans : {csv_val_path}")

CSV des images par classe dans la validation sauvegardé dans : val_class_counts.csv


## **2. Modèle**

In [7]:
val_class_counts = pd.read_csv("val_class_counts.csv")

weights = 1.0 / val_class_counts["num_images"].values

In [ ]:
class ResnetFineTune (nn.Module):
    def __init__(self, num_classes, 
                 dropout, 
                 freeze = False):
        
        super(ResnetFineTune, self).__init__()

        self.resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        if freeze:
            for param in self.resnet.parameters():
                param.requires_grad = False

            for param in self.resnet.layer4.parameters():
                param.requires_grad = True

        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, num_classes),
        )

    def forward (self, x):
        return self.resnet(x)
    
    def inference(self, x):
        self.eval()
        with torch.no_grad():
            logits = self(x)
            probs = torch.softmax(logits, dim=1)
            pred_class = torch.argmax(probs, dim=1).item()
        
        return pred_class

In [ ]:
model = ResnetFineTune(NUM_CLASSES, DROPOUT)
model.to(DEVICE)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

In [14]:
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np

def create_model(num_classes: int) -> nn.Module:

    model = models.efficientnet_b0(weights="IMAGENET1K_V1") #mettre b3 si ca marche
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, num_classes),
    )
    return model

model = create_model(num_classes)
model.to(DEVICE)

val_class_counts = pd.read_csv("val_class_counts.csv")
weights = 1.0 / np.sqrt(val_class_counts["num_images"].values)


criterion = nn.CrossEntropyLoss(label_smoothing=0.1, weight=torch.tensor(weights, dtype=torch.float32).to(DEVICE))

# --- Variante ---
# pip install torchmetrics
# from torchmetrics.classification import MulticlassFocalLoss
# criterion = MulticlassFocalLoss(num_classes=num_classes, alpha=0.25, gamma=2.0)

# FocalLoss peut être redondante avec l'utilisation d'un WeightedRandomSampler
# Label Smoothing dans la CrossEntropyLoss ? A 0.1 par exemple

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

# scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Usage de OneCycleLR
steps_per_epoch = len(train_loader)

scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6
)

## **3. F1-score**

In [15]:
import numpy as np

def compute_f1(all_labels, all_preds, num_classes):
    f1_per_class = []

    for cls in range(num_classes):
        # True Positives
        TP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) == cls))
        # False Positives
        FP = np.sum((np.array(all_preds) == cls) & (np.array(all_labels) != cls))
        # False Negatives
        FN = np.sum((np.array(all_preds) != cls) & (np.array(all_labels) == cls))

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall    = TP / (TP + FN) if (TP + FN) > 0 else 0

        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        f1_per_class.append(f1)

    # F1 macro : moyenne des classes
    f1_macro = np.mean(f1_per_class)
    return f1_macro, f1_per_class

## **4. Fonctions de training et validation**

In [ ]:
def train_one_epoch():
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    for images, labels in tqdm(train_loader):
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # 🔹 Calcul F1 avec ta fonction existante
    f1_macro, f1_per_class = compute_f1(all_labels, all_preds, NUM_CLASSES)

    # 🔹 Affichage
    # print(f"Train F1 macro: {f1_macro:.4f}")

    return total_loss / len(train_loader), correct / total, f1_macro, f1_per_class


In [ ]:
import torch

def validate():
    model.eval()
    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calcul F1 manuel par classe
    f1_macro, f1_per_class = compute_f1(all_labels, all_preds, NUM_CLASSES)

    return (total_loss / len(val_loader), f1_macro, f1_per_class, np.array(all_preds), np.array(all_labels))

## **5. Entrainement**

**Entrainement**

In [ ]:
import csv
import pandas as pd
from sklearn.metrics import confusion_matrix

best_f1 = 0.0
best_model_path = "best_model.pth"

csv_file = "training_log.csv"
fieldnames = ['epoch', 'train_loss', 'train_acc', 'train_f1_macro',
              'val_loss', 'val_f1_macro']


loss_train, loss_val = [], []

for epoch in range(NUM_EPOCHS):

    # ===== TRAIN =====
    train_loss, train_acc, train_f1_macro, train_f1_per_class = train_one_epoch()
    loss_train.append(train_loss)
    # scheduler.step()

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Train F1 Macro: {train_f1_macro:.4f}")

    # ===== VALIDATION =====
    val_loss, val_f1_macro, val_f1_per_class, val_preds, val_labels = validate()
    loss_val.append(val_loss)
    
    print(f"Val Loss: {val_loss:.4f}")
    print(f"Val F1 Macro: {val_f1_macro:.4f}")



    # ===== BEST MODEL =====
    if val_f1_macro > best_f1:

        best_f1 = val_f1_macro
        torch.save(model.state_dict(), best_model_path)

        print(f" Nouveau meilleur modèle sauvegardé ! F1 Macro = {best_f1:.4f}")


       

100%|██████████| 201/201 [02:12<00:00,  1.52it/s]


Epoch 1/25
Train Loss: 2.7960 | Train Acc: 0.2482
Train F1 Macro: 0.1901


Val Loss: 4.9647
Val F1 Macro: 0.1340
 Nouveau meilleur modèle sauvegardé ! F1 Macro = 0.1340
 Metrics par classe sauvegardées


100%|██████████| 201/201 [02:14<00:00,  1.50it/s]


Epoch 2/25
Train Loss: 1.5489 | Train Acc: 0.4873
Train F1 Macro: 0.4256


...

Val Loss: 3.3530
Val F1 Macro: 0.6466
 Nouveau meilleur modèle sauvegardé ! F1 Macro = 0.6466
 Metrics par classe sauvegardées


100%|██████████| 201/201 [02:14<00:00,  1.50it/s]


Epoch 22/25
Train Loss: 0.7714 | Train Acc: 0.9017
Train F1 Macro: 0.8976


Val Loss: 3.3605
Val F1 Macro: 0.6376


 40%|███▉      | 80/201 [00:53<01:19,  1.51it/s]

In [ ]:
plt.plot(loss_train, 'r' , marker='o', linestyle='-', label = 'loss_train')
plt.plot(loss_val, 'b' , marker='o', linestyle='-', label = 'loss_test')
plt.xlabel('Epoch')
plt.ylabel('Loss fonction')
plt.grid()
plt.legend()
plt.show()